# **Initialization**

In [1]:
print('Start')

Start


In [2]:
%load_ext autoreload
#%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import pulp
import vrplib
import re
import sys
import os
import gc
import glob
import contextlib
import modified_didppy as m_dp
import time as pytime

# **Data**

In [3]:
def read_formated_data(file_path):
    """
    Reads a Solomon format .txt file using vrplib and returns a dictionary 
    formatted for CVRPTW LP Relaxation and DIDP models.
    
    Ensures all numerical data (demand, time windows, service times, costs, capacity) 
    are returned as floats.
    """
    # 1. Read instance using vrplib
    # instance_format='solomon' ensures correct parsing of sections
    instance = vrplib.read_instance(file_path, instance_format='solomon')

    # 2. Extract Data & Cast to Float
    # 'edge_weight' is the distance matrix computed by vrplib
    travel_cost = instance['edge_weight'].astype(float).tolist()
    
    # 'node_coord' is available if you ever need it, but we use the pre-calc weights
    num_locations = len(instance['node_coord'])

    # 3. Return Bundle
    return {
        'num_locations': num_locations,
        'num_vehicles': int(instance.get('vehicles', 25)), 
        'capacity': float(instance['capacity']),
        
        # Cast demand to float list
        'demand': instance['demand'].astype(float).tolist(),
        
        # Cast Time Windows to float list
        # Col 0 is ready_time (earliest arrival), Col 1 is due_date (latest arrival)
        'ready_time': instance['time_window'][:, 0].astype(float).tolist(),
        'due_date': instance['time_window'][:, 1].astype(float).tolist(),
        
        # Cast Service Time to float list
        'service_time': instance['service_time'].astype(float).tolist(),
        
        # Use the pre-computed edge weights from vrplib
        'travel_cost': travel_cost
    }
    
def get_best_known_solution(instance_file_path, bk_dict=None):
    """
    1. Tries to find the cost in the provided dictionary (bk_dict).
       - Normalizes filename to lowercase and removes extension to match dict keys.
    2. If not found or no dict provided, looks for a corresponding .sol file.
    """
    
    # --- 1. Dictionary Lookup ---
    if bk_dict:
        # Extract filename (e.g., "C1_2_1.TXT")
        filename = os.path.basename(instance_file_path)
        # Remove extension and convert to lowercase (e.g., "c1_2_1")
        key_name = os.path.splitext(filename)[0].lower()
        
        if key_name in bk_dict:
            return bk_dict[key_name]

    # --- 2. Fallback: .sol file lookup ---
    base_path = instance_file_path.rsplit('.', 1)[0]
    sol_path = base_path + '.sol'
    
    if os.path.exists(sol_path):
        try:
            solution = vrplib.read_solution(sol_path)
            return solution.get('cost', None) 
        except Exception as e:
            # print(f"Warning: Could not read solution file {sol_path}: {e}")
            return None
            
    return None

# These variable names match what is typically expected by your DIDP/LP models
current_num_locations = 2
current_num_vehicles  = 2
current_capacity      = 10.0
current_cust_demand   = [0.0, 0.0]
current_avail_time    = [0.0, 0.0]
current_due_date      = [1000.0, 1000.0]
current_serve_time  = [0.0, 0.0]
current_travel_cost   = [[[0.0] for i in range(1,3)] for row in range(1,3)]

# Directory containing the VRP instances (Update this path)
# Note: Ensure this folder contains your .vrp files (e.g., A-n33-k5.vrp)
Solomon_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon"
HG_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\homberger_200_customer_instances"
folder_path = HG_folder_path
#folder_path = Solomon_folder_path
# Get all .vrp files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select random instances (or all)
num_instances_to_test = 1000
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
#print("Selected Instances:")
#for f in selected_files:
#   print(f" - {os.path.basename(f)}")
#print("-" * 50)

best_known_costs_HG_200 = {
    # 200 Customers
    "c1_2_1": 2704.57, "c1_2_2": 2917.89, "c1_2_3": 2707.35, "c1_2_4": 2643.31, "c1_2_5": 2702.05, 
    "c1_2_6": 2701.04, "c1_2_7": 2701.04, "c1_2_8": 2775.48, "c1_2_9": 2687.83, "c1_2_10": 2643.51,
    "c2_2_1": 1931.44, "c2_2_2": 1863.16, "c2_2_3": 1775.08, "c2_2_4": 1703.43, "c2_2_5": 1878.85, 
    "c2_2_6": 1857.35, "c2_2_7": 1849.46, "c2_2_8": 1820.53, "c2_2_9": 1830.05, "c2_2_10": 1806.58,
    "r1_2_1": 4784.11, "r1_2_2": 4039.86, "r1_2_3": 3381.96, "r1_2_4": 3057.81, "r1_2_5": 4107.86, 
    "r1_2_6": 3583.14, "r1_2_7": 3150.11, "r1_2_8": 2951.99, "r1_2_9": 3760.58, "r1_2_10": 3301.18,
    "r2_2_1": 4483.16, "r2_2_2": 3621.20, "r2_2_3": 2880.62, "r2_2_4": 1981.29, "r2_2_5": 3366.79, 
    "r2_2_6": 2913.03, "r2_2_7": 2451.14, "r2_2_8": 1849.87, "r2_2_9": 3092.04, "r2_2_10": 2654.97,
    "rc1_2_1": 3602.80, "rc1_2_2": 3249.05, "rc1_2_3": 3008.33, "rc1_2_4": 2851.68, "rc1_2_5": 3371.00, 
    "rc1_2_6": 3324.80, "rc1_2_7": 3189.32, "rc1_2_8": 3083.93, "rc1_2_9": 3081.13, "rc1_2_10": 3000.30,
    "rc2_2_1": 3099.53, "rc2_2_2": 2825.24, "rc2_2_3": 2601.87, "rc2_2_4": 2038.56, "rc2_2_5": 2911.46, 
    "rc2_2_6": 2873.12, "rc2_2_7": 2525.83, "rc2_2_8": 2292.53, "rc2_2_9": 2175.04, "rc2_2_10": 2015.60
    }

Found 60 files. Selected 60 for testing.


# **DIDP model**

In [4]:
def creation_of_didp_model_function():
    num_locations = current_num_locations
    num_vehicles = current_num_vehicles
    q = current_capacity
    cust_demand = current_cust_demand
    avail_time = current_avail_time
    due_date = current_due_date
    serve_time = current_serve_time
    travel_cost = current_travel_cost
    
    # =====================================================================================
    # DIDP Model Definition
    # =====================================================================================
    model = m_dp.Model(float_cost=True)

    # Object types for customers/locations and vehicles
    customer = model.add_object_type(number=num_locations)
    vehicle = model.add_object_type(number=num_vehicles)

    # -------------------- State Variables --------------------
    # Set of unvisited customers
    unvisited_locations = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))

    # Per-vehicle state variables, stored in Python lists for easy access
    vehicle_locations = [
    model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_loads = [
    model.add_float_var(target=0, name=f"load_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_times = [
    model.add_float_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
    for v in range(num_vehicles)
    ]


    # -------------------- Tables of Constants --------------------
    demand = model.add_float_table(cust_demand)
    ready_time = model.add_float_table(avail_time)
    due_time = model.add_float_table(due_date)
    service_time = model.add_float_table(serve_time)
    travel_time = model.add_float_table(travel_cost)

    # -------------------- Transitions --------------------
    for v in range(num_vehicles):
        for j in range(1, num_locations):
            # Expression for arrival time at customer j with vehicle v
            arrival_time = m_dp.max(
            vehicle_times[v] + travel_time[vehicle_locations[v], j],
            ready_time[j]
            )

            departure_time = arrival_time + service_time[j]

            visit_transition = m_dp.Transition(
                name=f"visit_{j}_with_vehicle_{v}",
                cost=travel_time[vehicle_locations[v], j] + m_dp.FloatExpr.state_cost(),
                preconditions=[
                    unvisited_locations.contains(j),
                    vehicle_loads[v] + demand[j] <= q,
                    arrival_time <= due_time[j],
                ],
                effects=[
                    (unvisited_locations, unvisited_locations.remove(j)),
                    (vehicle_locations[v], j),
                    (vehicle_loads[v], vehicle_loads[v] + demand[j]),
                    (vehicle_times[v], departure_time),
                ],
            )
            model.add_transition(visit_transition)

    # Transitions for each vehicle to return to the depot after all customers are served
    for v in range(num_vehicles):
        return_to_depot_transition = m_dp.Transition(
            name=f"return_vehicle_{v}_to_depot",
            cost=travel_time[vehicle_locations[v], 0] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited_locations.is_empty(), vehicle_locations[v] != 0],
            effects=[(vehicle_locations[v], 0)],
        )
        model.add_transition(return_to_depot_transition)

    # --- 1. Global Capacity Constraint (The Efficient "Cut") ---
    # Logic: Total Capacity Available >= Total Demand Remaining
    # Summing variables in Python creates a DIDP expression automatically
    total_current_load = sum(vehicle_loads) 
    total_fleet_capacity = num_vehicles * q
    
    # demand[unvisited_locations] automatically sums the weight of items in the set
    model.add_state_constr(
        (total_fleet_capacity - total_current_load) >= demand[unvisited_locations]
    )
    
    # -------------------- Base Case --------------------
    # All customers visited AND all vehicles are at the depot
    base_conditions = [unvisited_locations.is_empty()]
    for v in range(num_vehicles):
        base_conditions.append(vehicle_locations[v] == 0)
    model.add_base_case(base_conditions)

    # =========================================================
    # Dual Bounds
    # =========================================================

    # --- Pre-computation of Min Edge Tables ---
    # min_from[i]: The cheapest cost to LEAVE node i
    min_from_val = [min(travel_cost[i][k] for k in range(num_locations) if k != i) for i in range(num_locations)]
    min_from = model.add_float_table(min_from_val)

    # min_to[j]: The cheapest cost to ENTER node j
    min_to_val = [min(travel_cost[k][j] for k in range(num_locations) if k != j) for j in range(num_locations)]
    min_to = model.add_float_table(min_to_val)

    # --- Dual Bound 1: Minimum Outgoing Edges ---
    # Logic: 
    # 1. We must leave every customer that is currently unvisited.
    # 2. Every vehicle that is currently NOT at the depot must leave its current location.
    lb_outgoing = min_from[unvisited_locations]  # Sum of min_from for all unvisited nodes
    for v in range(num_vehicles):
        # If vehicle v is at a customer (location != 0), it must leave that customer eventually.
        # We add the min cost to leave its current location.
        lb_outgoing += (vehicle_locations[v] != 0).if_then_else(min_from[vehicle_locations[v]], 0.0)
    model.add_dual_bound(lb_outgoing)
    
    # --- Dual Bound 2: Minimum Incoming Edges ---
    # Logic:
    # 1. We must enter every customer that is currently unvisited.
    # 2. Every vehicle that is currently NOT at the depot must eventually return (enter) the depot.
    lb_incoming = min_to[unvisited_locations] # Sum of min_to for all unvisited nodes
    for v in range(num_vehicles):
        # If vehicle v is out working (location != 0), it must return to depot (enter node 0).
        lb_incoming += (vehicle_locations[v] != 0).if_then_else(min_to[0], 0.0)
    model.add_dual_bound(lb_incoming)

    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_locations": unvisited_locations,
        "vehicle_locations": vehicle_locations,
        "vehicle_loads": vehicle_loads,
        "vehicle_times": vehicle_times,
        "distance_matrix": travel_cost,
        "demand": cust_demand,
        "due_time": due_date,
        "ready_time": avail_time,
        "service_time": serve_time,
        "capacity": q,
        "num_vehicles": num_vehicles,
        "num_locations": num_locations
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Run single dual bound model on selected instances**

In [5]:
# =============================================================================
# 1. SETUP: Directories & Inputs
# =============================================================================

# Paths to your datasets
Solomon_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon"
HG_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\homberger_200_customer_instances"

# Input CSV containing the instances to run
input_csv_path = "CVRPTW_single_dual_bound_results_10s_lim.csv"

# =============================================================================
# 2. READ INSTANCES & LOCATE FILES
# =============================================================================

print(f"Reading target instances from: {input_csv_path}")
try:
    df_input = pd.read_csv(input_csv_path)
    # Extract the 'Instance' column
    if 'Instance' in df_input.columns:
        all_target_instances = df_input['Instance'].dropna().astype(str).tolist()
    else:
        # Fallback if first column is the instance name but labeled differently
        all_target_instances = df_input.iloc[:, 0].dropna().astype(str).tolist()
    
    print(f"Found {len(all_target_instances)} instances in CSV.")

except Exception as e:
    print(f"Error reading input CSV: {e}")
    all_target_instances = []

# Function to find file in multiple folders (case-insensitive)
def get_full_path(filename, folder_paths):
    for folder in folder_paths:
        for name_variant in [filename, filename.lower(), filename.upper()]:
            full_path = os.path.join(folder, name_variant)
            if os.path.exists(full_path):
                return full_path
    return None

files_to_run = []
folders = [Solomon_folder_path, HG_folder_path]

print("Locating files on disk...")
found_count = 0
for inst_name in all_target_instances:
    path = get_full_path(inst_name, folders)
    if path:
        files_to_run.append(path)
        found_count += 1
    else:
        print(f"Warning: Could not locate file for '{inst_name}' in specified folders.")

print(f"Successfully located {found_count} out of {len(all_target_instances)} files.")

Reading target instances from: CVRPTW_single_dual_bound_results_10s_lim.csv
Found 10 instances in CSV.
Locating files on disk...
Successfully located 10 out of 10 files.


In [7]:
# =============================================================================
# 2. RESUME LOGIC & EXECUTION
# =============================================================================

output_csv_name = "CVRPTW_1T_single_dual_bound_selected_results_1800s_lim.csv"
processed_instances = []

# Check if CSV exists to resume
if os.path.exists(output_csv_name):
    try:
        df_existing = pd.read_csv(output_csv_name)
        if "Instance" in df_existing.columns:
            processed_instances = df_existing["Instance"].tolist()
            print(f"Found existing results file. {len(processed_instances)} instances already processed.")
    except Exception as e:
        print(f"Warning: Could not read existing CSV ({e}). Starting fresh.")

# Filter out files that are already done
# Note: 'files_to_run' is assumed to be defined in your previous cell
remaining_files = [f for f in files_to_run if os.path.basename(f) not in processed_instances]
print(f"Starting execution on {len(remaining_files)} remaining instances (Total selected: {len(files_to_run)}).")

# =============================================================================
# 3. EXECUTION LOOP
# =============================================================================

time_limit_seconds = 1800

for i, file_path in enumerate(remaining_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(remaining_files)}] Processing: {instance_name}")
    
    # --- 1. Get Best Known Cost ---
    best_known_cost = get_best_known_solution(file_path, bk_dict=best_known_costs_HG_200)

    result_entry = {}
    
    try:
        # --- 2. Read Data ---
        data = read_formated_data(file_path)

        # Update globals required for the model function
        current_num_locations = data['num_locations']
        current_num_vehicles  = data['num_vehicles']
        current_capacity      = data['capacity']
        current_cust_demand   = data['demand']
        current_avail_time    = data['ready_time']
        current_due_date      = data['due_date']
        current_serve_time    = data['service_time']
        current_travel_cost   = data['travel_cost']
        
        # --- 3. Create Model ---
        model, state_data = creation_of_didp_model_function()
        
        # --- 4. Run Solver ---
        t_start = pytime.time()
        
        solver = m_dp.CABS(
            model,
            quiet=False,
            time_limit=time_limit_seconds
        )
        
        solution = solver.search()
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- 5. Parse Results ---
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = float('inf')
            status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        # Gap Calculation
        gap = "N/A"
        if best_known_cost is not None and cost != float('inf') and cost != "Inf":
            try:
                gap_val = ((cost - best_known_cost) / best_known_cost) * 100
                gap = f"{gap_val:.2f}%"
            except:
                gap = "Error"

        print(f"   -> My Cost: {cost} | BKS: {best_known_cost} | Gap: {gap}")
        print(f"   -> Time: {duration:.2f}s | Nodes Exp: {nodes_exp} | Status: {status}")

        result_entry = {
            "Instance": instance_name,
            "Best Known Cost": best_known_cost,
            "Cost": cost,
            "Gap to BKS": gap,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        }

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        result_entry = {
            "Instance": instance_name,
            "Best Known Cost": best_known_cost,
            "Cost": "Error",
            "Gap to BKS": "N/A",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error",
            "Infeasibility": str(e)
        }

    # --- 6. Save Immediately (Append Mode) ---
    df_single_result = pd.DataFrame([result_entry])
    
    # If file doesn't exist, write header. If it exists, append without header.
    if not os.path.exists(output_csv_name):
        df_single_result.to_csv(output_csv_name, index=False)
    else:
        df_single_result.to_csv(output_csv_name, mode='a', header=False, index=False)

print("\n" + "="*50)
print("Batch Execution Complete.")
print(f"Results saved to: {output_csv_name}")

Starting execution on 10 remaining instances (Total selected: 10).

[1/10] Processing: C104.txt
   -> My Cost: 1444.7485802926383 | BKS: 822.9 | Gap: 75.57%
   -> Time: 1800.60s | Nodes Exp: 61889 | Status: False (Time Limit)

[2/10] Processing: C201.txt
   -> My Cost: 1569.3074082735395 | BKS: 589.1 | Gap: 166.39%
   -> Time: 1800.17s | Nodes Exp: 65597 | Status: False (Time Limit)

[3/10] Processing: R104.txt
   -> My Cost: 1531.8005085552288 | BKS: 971.5 | Gap: 57.67%
   -> Time: 1800.14s | Nodes Exp: 71240 | Status: False (Time Limit)

[4/10] Processing: R203.txt
   -> My Cost: 1467.5311525751467 | BKS: 870.8 | Gap: 68.53%
   -> Time: 1800.11s | Nodes Exp: 61568 | Status: False (Time Limit)

[5/10] Processing: RC104.txt
   -> My Cost: 1731.5377608485553 | BKS: 1132.3 | Gap: 52.92%
   -> Time: 1800.14s | Nodes Exp: 67502 | Status: False (Time Limit)

[6/10] Processing: C1_2_10.TXT
   -> My Cost: 4513.5107968426555 | BKS: 2643.51 | Gap: 70.74%
   -> Time: 1800.31s | Nodes Exp: 9108 |